# 🔠 Letterboxed Solver (Ruby)

Finds the **minimum number of words** to solve the NYT Letterboxed puzzle.

### Rules
- 12 letters on 4 sides (3 per side)
- Consecutive letters in a word **cannot** come from the same side
- The **last letter** of each word must be the **first letter** of the next
- All 12 letters must be used across all words

In [ ]:
# ============================================================
# CELL 1: Configure your puzzle here
# ============================================================
# Enter the 4 sides of today's puzzle (3 letters each, order matters)
# Top, Right, Bottom, Left — clockwise from the top

SIDES = [
  %w[n y e],   # top
  %w[a t h],   # right
  %w[i o s],   # bottom
  %w[r c l]    # left
]

# Path to a word list file (one word per line)
# macOS/Linux built-in:  /usr/share/dict/words
# Download a better one: https://github.com/dwyl/english-words
WORD_LIST_PATH = '/usr/share/dict/words'

puts "Puzzle sides:"
SIDES.each_with_index { |s, i| puts "  #{%w[Top Right Bottom Left][i]}: #{s.join(' ').upcase}" }
puts "All letters: #{SIDES.flatten.join.upcase}"

In [ ]:
# ============================================================
# CELL 2: Build lookup structures
# ============================================================
require 'set'

# Map each letter → which side index it belongs to (0–3)
$letter_to_side = {}
SIDES.each_with_index do |side, i|
  side.each { |letter| $letter_to_side[letter] = i }
end

# Map each puzzle letter → a unique bit position (0–11)
puzzle_letters = SIDES.flatten
$letter_to_bit  = puzzle_letters.each_with_index.to_h
$puzzle_set     = puzzle_letters.to_set
$full_mask      = (1 << 12) - 1  # all 12 bits set = all letters covered

puts "Letter → side:  #{$letter_to_side}"
puts "Letter → bit:   #{$letter_to_bit}"
puts "Full mask (bin): #{$full_mask.to_s(2).rjust(12, '0')}"

In [ ]:
# ============================================================
# CELL 3: Load + filter valid words
# ============================================================

# A word is valid for Letterboxed if:
#   1. Length >= 3
#   2. Every letter appears in the puzzle
#   3. No two consecutive letters are on the same side
def letterboxed_valid?(word)
  return false if word.length < 3
  chars = word.chars
  return false unless chars.all? { |c| $puzzle_set.include?(c) }
  chars.each_cons(2).all? { |a, b| $letter_to_side[a] != $letter_to_side[b] }
end

puts "Loading word list from #{WORD_LIST_PATH}..."

raw_words = File.readlines(WORD_LIST_PATH)
             .map  { |w| w.strip.downcase }
             .select { |w| w.match?(/\A[a-z]+\z/) }

$valid_words = raw_words.select { |w| letterboxed_valid?(w) }

puts "Total words in list : #{raw_words.length}"
puts "Valid for this puzzle: #{$valid_words.length}"
puts "Sample: #{$valid_words.first(10).join(', ')}"

In [ ]:
# ============================================================
# CELL 4: Precompute bitmasks + group words by starting letter
# ============================================================

# Each word entry: { word, start, last, mask }
# mask = bitmask of which puzzle letters the word covers
$word_entries = $valid_words.map do |w|
  mask = w.chars.reduce(0) { |m, c| m | (1 << $letter_to_bit[c]) }
  { word: w, start: w[0], last: w[-1], mask: mask }
end

# Group by starting letter for O(1) lookup when chaining words
$by_start = Hash.new { |h, k| h[k] = [] }
$word_entries.each { |e| $by_start[e[:start]] << e }

puts "Words indexed by starting letter:"
$by_start.sort.each { |letter, words| puts "  #{letter.upcase}: #{words.length} words" }

In [ ]:
# ============================================================
# CELL 5: Bitmask BFS solver
# ============================================================
# State = [last_letter, coverage_bitmask]
# We find the minimum-length word chain that covers all 12 letters.
#
# BFS guarantees the first time we reach full_mask is the minimum.
# We collect ALL solutions at that minimum depth before stopping.

def solve_letterboxed(max_words: 5, max_solutions: 20)
  # visited[last_letter][mask] = true → already explored this state at min depth
  visited = Hash.new { |h, k| h[k] = {} }

  # Queue entries: { last:, mask:, chain: }
  # Seed with every valid word as a starting point
  queue = []
  $word_entries.each do |e|
    queue << { last: e[:last], mask: e[:mask], chain: [e[:word]] }
  end

  solutions  = []
  found_depth = nil

  until queue.empty?
    cur = queue.shift
    last, mask, chain = cur[:last], cur[:mask], cur[:chain]
    depth = chain.length

    # Once we have solutions, only keep collecting at the same depth
    break if found_depth && depth > found_depth
    break if depth > max_words
    break if solutions.length >= max_solutions

    # Skip already-explored states (we only need one path per state for minimum)
    next if visited[last][mask]
    visited[last][mask] = true

    if mask == $full_mask
      solutions << chain
      found_depth = depth
      next
    end

    # Extend chain: next word must start with `last`
    ($by_start[last] || []).each do |e|
      queue << { last: e[:last], mask: mask | e[:mask], chain: chain + [e[:word]] }
    end
  end

  solutions
end

puts "Solver defined. Run the next cell to solve."

In [ ]:
# ============================================================
# CELL 6: Run the solver!
# ============================================================

puts "Solving..."
t0 = Time.now
solutions = solve_letterboxed(max_words: 5, max_solutions: 20)
elapsed = Time.now - t0

if solutions.empty?
  puts "No solution found within the word/depth limits."
  puts "Try: a larger word list, or increase max_words."
else
  min_words = solutions.first.length
  puts "✓ Solved in #{elapsed.round(2)}s"
  puts "Best solution uses #{min_words} word(s) — showing up to #{solutions.length} variants:\n\n"

  solutions.each_with_index do |sol, i|
    # Show coverage per word
    covered = ''
    detail = sol.map do |w|
      new_letters = w.chars.select { |c| !covered.include?(c) }
      covered += new_letters.join
      "#{w} (+#{new_letters.join.upcase})"
    end
    puts "#{(i+1).to_s.rjust(2)}. #{sol.join(' → ')}\n    #{detail.join(' | ')}"
  end
end

In [ ]:
# ============================================================
# CELL 7: (Optional) Verify a specific solution manually
# ============================================================

def verify_solution(words)
  puts "Verifying: #{words.join(' → ')}"
  errors = []

  words.each_with_index do |word, i|
    # Rule: valid letterboxed word
    unless letterboxed_valid?(word)
      errors << "'#{word}' is not valid (same-side consecutive letters or unknown letters)"
    end

    # Rule: chaining — last letter of word[i] == first letter of word[i+1]
    if i < words.length - 1
      unless word[-1] == words[i+1][0]
        errors << "Chain break: '#{word}' ends with '#{word[-1]}' but '#{words[i+1]}' starts with '#{words[i+1][0]}'"
      end
    end
  end

  covered = words.join.chars.to_set
  missing = $puzzle_set - covered
  errors << "Missing letters: #{missing.to_a.join.upcase}" unless missing.empty?

  if errors.empty?
    puts "✓ Valid solution! All #{$puzzle_set.length} letters covered."
  else
    puts "✗ Invalid:"
    errors.each { |e| puts "  - #{e}" }
  end
end

# Example — replace with your words:
verify_solution(['stone', 'early'])  # just an example, adjust to your puzzle